In [1]:
import os
import json
import pandas as pd
import numpy as np

In [2]:
def load_results(path, task_keys=None, task_metrics=None, groups=None):
    files = os.listdir(path)
    if task_keys is None:
        results = []
    else:
        results = {}
    for filename in files:
        if filename.startswith(".") or "results_only" in filename:
            continue
        buf_path = os.path.join(path, filename)
        with open(buf_path, "r") as f:
            data = json.load(f)
        if task_keys is None:
            results.append(data["results"])
        else:
            if groups is None:
                for idx, task in enumerate(task_keys):
                    buf_res = data["results"].get(task)
                    if buf_res is not None:
                        results[task] = f"{np.round(buf_res[task_metrics[idx][0]] * 100, 2)}±{np.round(buf_res[task_metrics[idx][1]] * 100, 2)}"
            else:
                gr_start = 0
                for idx_gr, group in enumerate(groups):
                    gr_name, gr_len = list(group.items())[0]
                    gr_end = gr_start+gr_len
                    gr_mean, gr_std = [], []
                    for idx in range(gr_start, gr_end):
                        task = task_keys[idx]
                        buf_res = data["results"].get(task)
                        gr_mean.append(buf_res[task_metrics[idx][0]])
                        gr_std.append(buf_res[task_metrics[idx][1]])
                    results[gr_name] = f"{np.round(np.mean(gr_mean) * 100, 2)}±{np.round(np.mean(gr_std) * 100, 2)}"
                    gr_start = gr_end
    return results

In [208]:
base_path = "../../harness_results/meta-llama"
model_pathes = [
    "Llama-3.2-1B/base-no-template/metrics.jsonl/meta-llama__Llama-3.2-1B",
    "Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000-no-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__databricks-dolly-15k_ift_1024_seed_42__checkpoint-2000",
    "Llama-3.2-1B/alpaca-cleaned_ift_1024_seed_42/checkpoint-7000-no-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__alpaca-cleaned_ift_1024_seed_42__checkpoint-7000",
    "Llama-3.2-1B/unnatural-instructions-full_ift_1024_seed_42/checkpoint-8000-no-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__unnatural-instructions-full_ift_1024_seed_42__checkpoint-8000",

    "Llama-3.2-1B/ultrachat_200k_flattened_ift_1024_seed_42/checkpoint-30000-no-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__ultrachat_200k_flattened_ift_1024_seed_42__checkpoint-30000",
    "Llama-3.2-1B/hf_dataset_sharegpt_ift_1024_seed_42/checkpoint-30000-no-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__hf_dataset_sharegpt_ift_1024_seed_42__checkpoint-30000",
    "Llama-3.2-1B/flan_collection_ift_1024_seed_42/checkpoint-30000-no-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__flan_collection_ift_1024_seed_42__checkpoint-30000",
    
    
    "Llama-3.2-1B-Instruct/base-no-template/metrics.jsonl/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Alpaca-Llama-3.2-1B",
    "Unnat-Llama-3.2-1B",

    "UC-Llama-3.2-1B",
    "SGpt-Llama-3.2-1B",
    "FLAN-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa", "gpqa_main_cot_zeroshot", "gsm8k_cot"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5 + [["exact_match,flexible-extract", "exact_match_stderr,flexible-extract"]]*2
#task_metrics = [["acc,none", "acc_stderr,none"]]*5 + [["exact_match,strict-match", "exact_match_stderr,strict-match"]]*2



In [95]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics)
    overall_res[model_name] = buf_res

In [96]:
#overall_res

In [97]:
pd.DataFrame.from_dict(overall_res).sort_index()

,Llama-3.2-1B,Dolly-Llama-3.2-1B,Alpaca-Llama-3.2-1B,Unnat-Llama-3.2-1B,UC-Llama-3.2-1B,SGpt-Llama-3.2-1B,FLAN-Llama-3.2-1B,Llama-3.2-1B-IT
arc_challenge,31.57±1.36,33.11±1.38,32.0±1.36,29.95±1.34,32.59±1.37,32.42±1.37,31.4±1.36,35.75±1.4
arc_easy,65.49±0.98,65.61±0.97,62.92±0.99,61.15±1.0,65.19±0.98,65.74±0.97,62.46±0.99,68.69±0.95
gpqa_main_cot_zeroshot,12.28±1.55,12.05±1.54,14.06±1.64,10.94±1.48,10.49±1.45,9.15±1.36,15.85±1.73,9.6±1.39
gsm8k_cot,7.13±0.71,5.31±0.62,6.14±0.66,4.93±0.6,6.97±0.7,6.44±0.68,5.38±0.62,33.97±1.3
hellaswag,47.74±0.5,49.2±0.5,50.21±0.5,48.89±0.5,48.52±0.5,48.21±0.5,48.19±0.5,45.19±0.5
mathqa,29.25±0.83,28.44±0.83,28.38±0.83,28.24±0.82,29.15±0.83,30.02±0.84,28.78±0.83,33.33±0.86
mmlu,30.89±0.39,30.19±0.39,31.08±0.39,30.57±0.39,29.91±0.38,31.12±0.39,26.28±0.37,45.41±0.41


In [114]:
base_path = "../../harness_results/meta-llama"
model_pathes = [
    "Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__databricks-dolly-15k_ift_1024_seed_42__checkpoint-2000",
    "Llama-3.2-1B/alpaca-cleaned_ift_1024_seed_42/checkpoint-7000-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__alpaca-cleaned_ift_1024_seed_42__checkpoint-7000",
    "Llama-3.2-1B/unnatural-instructions-full_ift_1024_seed_42/checkpoint-8000-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__unnatural-instructions-full_ift_1024_seed_42__checkpoint-8000",

    "Llama-3.2-1B/ultrachat_200k_flattened_ift_1024_seed_42/checkpoint-30000-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__ultrachat_200k_flattened_ift_1024_seed_42__checkpoint-30000",
    "Llama-3.2-1B/hf_dataset_sharegpt_ift_1024_seed_42/checkpoint-30000-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__hf_dataset_sharegpt_ift_1024_seed_42__checkpoint-30000",
    "Llama-3.2-1B/flan_collection_ift_1024_seed_42/checkpoint-30000-template/__home__jovyan__gkuzmin__rmt_it__llama_non_it_exps__meta-llama__Llama-3.2-1B__flan_collection_ift_1024_seed_42__checkpoint-30000",
    
    
    "Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Dolly-CT-Llama-3.2-1B",
    "Alpaca-CT-Llama-3.2-1B",
    "Unnat-CT-Llama-3.2-1B",

    "UC-CT-Llama-3.2-1B",
    "SGpt-CT-Llama-3.2-1B",
    "FLAN-CT-Llama-3.2-1B",
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa", "gpqa_main_cot_zeroshot", "gsm8k_cot"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5 + [["exact_match,flexible-extract", "exact_match_stderr,flexible-extract"]]*2


In [115]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics)
    overall_res[model_name] = buf_res

In [116]:
#overall_res

In [117]:
pd.DataFrame.from_dict(overall_res).sort_index()

,Dolly-CT-Llama-3.2-1B,Alpaca-CT-Llama-3.2-1B,Unnat-CT-Llama-3.2-1B,UC-CT-Llama-3.2-1B,SGpt-CT-Llama-3.2-1B,FLAN-CT-Llama-3.2-1B,Llama-CT-3.2-1B-IT
arc_challenge,29.44±1.33,29.35±1.33,29.52±1.33,28.84±1.32,28.33±1.32,32.76±1.37,31.91±1.36
arc_easy,61.45±1.0,54.8±1.02,57.45±1.01,57.53±1.01,53.45±1.02,63.55±0.99,56.86±1.02
gpqa_main_cot_zeroshot,20.98±1.93,22.32±1.97,23.44±2.0,24.78±2.04,22.77±1.98,22.32±1.97,20.76±1.92
gsm8k_cot,3.71±0.52,0.15±0.11,3.56±0.51,3.71±0.52,0.0±0.0,4.93±0.6,33.36±1.3
hellaswag,45.72±0.5,46.79±0.5,47.43±0.5,46.98±0.5,45.95±0.5,46.6±0.5,44.61±0.5
mathqa,28.01±0.82,24.42±0.79,25.73±0.8,25.09±0.79,24.39±0.79,29.51±0.83,23.82±0.78
mmlu,23.04±0.35,23.13±0.36,27.92±0.38,25.41±0.37,22.95±0.35,27.53±0.37,23.77±0.36


In [24]:
buf_path = os.path.join(base_path, model_pathes[0])
load_results(buf_path, task_keys, task_metrics)

[{'mmlu': 0.30893035180173767}, {'arc_challenge': 0.31569965870307165}, {'arc_easy': 0.6548821548821548}, {'hellaswag': 0.47739494124676357}, {'mathqa': 0.29246231155778896}]


## Extract OpenLM Leaderboard results

In [496]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    "rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-CT-Llama-3.2-1B",
    "SGPT-CT-Llama-3.2-1B",
    "Flan-CT-Llama-3.2-1B",

    "ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    "ARMT-16-64-8s-DollyMix05-CT-Llama-3.2-1B-IT",
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [497]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [498]:
#overall_res

In [499]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()

In [500]:
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [501]:
res_df

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,SGPT-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B,ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-DollyMix05-CT-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.8±2.77,29.43±2.79,30.24±2.79,29.49±2.77,29.78±2.78,30.32±2.8,30.21±2.82,32.63±2.89,30.67±2.84,30.52±2.83,32.39±2.87,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,26.26±3.14,24.75±3.07,25.25±3.1,26.26±3.14,26.77±3.15,27.78±3.19,19.7±2.83,27.27±3.17,27.78±3.19,29.8±3.26,25.76±3.12,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,26.01±1.88,25.09±1.86,26.01±1.88,27.29±1.91,26.37±1.89,27.11±1.9,23.99±1.83,25.09±1.86,25.64±1.87,23.99±1.83,22.53±1.79,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,24.11±2.02,27.23±2.11,24.33±2.03,23.88±2.02,23.66±2.01,25.89±2.07,24.33±2.03,23.88±2.02,29.69±2.16,29.46±2.16,27.46±2.11,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,13.49±1.47,19.41±1.7,9.61±1.27,9.61±1.27,8.69±1.21,9.8±1.28,11.28±1.36,19.22±1.7,14.23±1.5,12.75±1.44,26.25±1.89,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.37±0.27,0.11±0.11,0.09±0.07,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.05±0.05,0.0±0.0,0.0±0.0,1.34±0.61,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.3±0.29,11.19±0.29,11.2±0.29,11.64±0.29,11.54±0.29,11.25±0.29,11.43±0.29,14.89±0.32,11.17±0.29,11.22±0.29,15.05±0.33,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,31.97±2.83,33.83±2.86,33.16±2.86,33.28±2.88,33.01±2.88,33.69±2.89,31.96±2.86,33.9±2.87,33.56±2.86,34.36±2.87,32.97±2.85,32.35±2.83


In [502]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]

In [503]:
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]

In [504]:
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))

/tmp/ipykernel_93986/1504875137.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [505]:
res_df.loc[8] = avg_res

In [506]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,SGPT-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B,ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-DollyMix05-CT-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.8±2.77,29.43±2.79,30.24±2.79,29.49±2.77,29.78±2.78,30.32±2.8,30.21±2.82,32.63±2.89,30.67±2.84,30.52±2.83,32.39±2.87,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,26.26±3.14,24.75±3.07,25.25±3.1,26.26±3.14,26.77±3.15,27.78±3.19,19.7±2.83,27.27±3.17,27.78±3.19,29.8±3.26,25.76±3.12,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,26.01±1.88,25.09±1.86,26.01±1.88,27.29±1.91,26.37±1.89,27.11±1.9,23.99±1.83,25.09±1.86,25.64±1.87,23.99±1.83,22.53±1.79,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,24.11±2.02,27.23±2.11,24.33±2.03,23.88±2.02,23.66±2.01,25.89±2.07,24.33±2.03,23.88±2.02,29.69±2.16,29.46±2.16,27.46±2.11,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,13.49±1.47,19.41±1.7,9.61±1.27,9.61±1.27,8.69±1.21,9.8±1.28,11.28±1.36,19.22±1.7,14.23±1.5,12.75±1.44,26.25±1.89,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.37±0.27,0.11±0.11,0.09±0.07,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.05±0.05,0.0±0.0,0.0±0.0,1.34±0.61,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.3±0.29,11.19±0.29,11.2±0.29,11.64±0.29,11.54±0.29,11.25±0.29,11.43±0.29,14.89±0.32,11.17±0.29,11.22±0.29,15.05±0.33,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,31.97±2.83,33.83±2.86,33.16±2.86,33.28±2.88,33.01±2.88,33.69±2.89,31.96±2.86,33.9±2.87,33.56±2.86,34.36±2.87,32.97±2.85,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,18.86,19.79,18.26,18.38,18.30,18.81,17.43,21.33,19.57,19.77,22.29,27.76


In [507]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_93986/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [508]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,SGPT-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B,ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-DollyMix05-CT-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44,30.75,29.80,29.43,30.24,29.49,29.78,30.32,30.21,32.63,30.67,30.52,32.39,33.46
1,gpqa_diamond,24.24,25.76,26.26,24.75,25.25,26.26,26.77,27.78,19.70,27.27,27.78,29.80,25.76,28.79
4,ifeval,10.17,12.20,13.49,19.41,9.61,9.61,8.69,9.80,11.28,19.22,14.23,12.75,26.25,50.28
5,math,0.12,0.32,0.37,0.11,0.09,0.00,0.00,0.00,0.00,0.05,0.00,0.00,1.34,5.53
6,mmlu_pro,11.79,11.05,11.30,11.19,11.20,11.64,11.54,11.25,11.43,14.89,11.17,11.22,15.05,16.17
7,musr,34.60,33.41,31.97,33.83,33.16,33.28,33.01,33.69,31.96,33.90,33.56,34.36,32.97,32.35
8,Avg. score (w. gpqa_diamond),18.73,18.92,18.86,19.79,18.26,18.38,18.30,18.81,17.43,21.33,19.57,19.77,22.29,27.76


## Extract OpenLM Leaderboard, LM-Only results

In [548]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    "rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    "data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    "data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    "data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    "data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    #"Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    "Flan-CT-Llama-3.2-1B",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    #"ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    "ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [549]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [550]:
#overall_res

In [551]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()

In [552]:
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [553]:
res_df

,task,Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Llama-3.2-1B,ARMT-16-64-2s-LM-31500-Llama-3.2-1B,ARMT-16-64-2s-LM-49500-Llama-3.2-1B,ARMT-16-64-4s-LM-19500-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,29.43±2.79,29.78±2.78,30.16±2.81,30.43±2.82,30.34±2.8,30.6±2.8,33.46±2.9
1,gpqa_diamond,24.24±3.05,24.75±3.07,26.77±3.15,26.77±3.15,25.25±3.1,27.27±3.17,29.29±3.24,28.79±3.23
2,gpqa_extended,24.54±1.84,25.09±1.86,26.37±1.89,26.56±1.89,26.19±1.88,26.01±1.88,26.92±1.9,25.09±1.86
3,gpqa_main,22.99±1.99,27.23±2.11,23.66±2.01,23.88±2.02,23.88±2.02,23.88±2.02,23.88±2.02,23.88±2.02
4,ifeval,10.17±1.3,19.41±1.7,8.69±1.21,13.49±1.47,13.86±1.49,14.6±1.52,11.46±1.37,50.28±2.15
5,math,0.12±0.12,0.11±0.11,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,5.53±1.56
6,mmlu_pro,11.79±0.29,11.19±0.29,11.54±0.29,11.58±0.29,11.79±0.29,11.86±0.29,11.25±0.29,16.17±0.34
7,musr,34.6±2.9,33.83±2.86,33.01±2.88,34.34±2.9,35.91±2.93,35.39±2.93,34.72±2.93,32.35±2.83


In [554]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]

In [555]:
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]

In [556]:
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))

/tmp/ipykernel_93986/1504875137.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [557]:
res_df.loc[8] = avg_res

In [558]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Llama-3.2-1B,ARMT-16-64-2s-LM-31500-Llama-3.2-1B,ARMT-16-64-2s-LM-49500-Llama-3.2-1B,ARMT-16-64-4s-LM-19500-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,29.43±2.79,29.78±2.78,30.16±2.81,30.43±2.82,30.34±2.8,30.6±2.8,33.46±2.9
1,gpqa_diamond,24.24±3.05,24.75±3.07,26.77±3.15,26.77±3.15,25.25±3.1,27.27±3.17,29.29±3.24,28.79±3.23
2,gpqa_extended,24.54±1.84,25.09±1.86,26.37±1.89,26.56±1.89,26.19±1.88,26.01±1.88,26.92±1.9,25.09±1.86
3,gpqa_main,22.99±1.99,27.23±2.11,23.66±2.01,23.88±2.02,23.88±2.02,23.88±2.02,23.88±2.02,23.88±2.02
4,ifeval,10.17±1.3,19.41±1.7,8.69±1.21,13.49±1.47,13.86±1.49,14.6±1.52,11.46±1.37,50.28±2.15
5,math,0.12±0.12,0.11±0.11,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,5.53±1.56
6,mmlu_pro,11.79±0.29,11.19±0.29,11.54±0.29,11.58±0.29,11.79±0.29,11.86±0.29,11.25±0.29,16.17±0.34
7,musr,34.6±2.9,33.83±2.86,33.01±2.88,34.34±2.9,35.91±2.93,35.39±2.93,34.72±2.93,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,19.79,18.30,19.39,19.54,19.91,19.55,27.76


In [559]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_93986/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [560]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Llama-3.2-1B,ARMT-16-64-2s-LM-31500-Llama-3.2-1B,ARMT-16-64-2s-LM-49500-Llama-3.2-1B,ARMT-16-64-4s-LM-19500-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44,29.43,29.78,30.16,30.43,30.34,30.60,33.46
1,gpqa_diamond,24.24,24.75,26.77,26.77,25.25,27.27,29.29,28.79
4,ifeval,10.17,19.41,8.69,13.49,13.86,14.60,11.46,50.28
5,math,0.12,0.11,0.00,0.00,0.00,0.00,0.00,5.53
6,mmlu_pro,11.79,11.19,11.54,11.58,11.79,11.86,11.25,16.17
7,musr,34.60,33.83,33.01,34.34,35.91,35.39,34.72,32.35
8,Avg. score (w. gpqa_diamond),18.73,19.79,18.30,19.39,19.54,19.91,19.55,27.76


## Extract OpenLM Leaderboard, ARMT results

In [733]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    "rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-98000/pytorch_model.bin/template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    "runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    "Flan-CT-Llama-3.2-1B",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B",
    "ARMT-16-64-5s-Flan-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    "ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    "ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B",
    "ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B",
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [734]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [735]:
#overall_res

In [736]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()

In [737]:
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [738]:
res_df

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B,ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT,ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.43±2.79,29.78±2.78,30.67±2.84,30.52±2.83,30.15±2.82,29.81±2.82,30.79±2.83,30.55±2.82,30.61±2.82,30.17±2.81,30.56±2.82,32.39±2.87,28.98±2.71,28.94±2.74,30.12±2.81,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.75±3.07,26.77±3.15,27.78±3.19,29.8±3.26,28.28±3.21,28.28±3.21,23.74±3.03,24.24±3.05,24.75±3.07,24.75±3.07,24.75±3.07,25.76±3.12,23.23±3.01,22.73±2.99,25.25±3.1,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.09±1.86,26.37±1.89,25.64±1.87,23.99±1.83,22.16±1.78,22.71±1.79,26.37±1.89,26.74±1.9,26.56±1.89,26.19±1.88,27.11±1.9,22.53±1.79,25.82±1.87,26.74±1.9,25.82±1.87,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.23±2.11,23.66±2.01,29.69±2.16,29.46±2.16,29.24±2.15,29.69±2.16,23.88±2.02,25.0±2.05,23.88±2.02,24.33±2.03,24.78±2.04,27.46±2.11,25.22±2.05,26.34±2.08,24.11±2.02,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,19.41±1.7,8.69±1.21,14.23±1.5,12.75±1.44,13.31±1.46,13.12±1.45,9.43±1.26,10.54±1.32,9.61±1.27,11.09±1.35,10.54±1.32,26.25±1.89,13.68±1.48,12.38±1.42,14.05±1.5,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.11±0.11,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,1.34±0.61,0.0±0.0,0.0±0.0,0.1±0.07,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.19±0.29,11.54±0.29,11.17±0.29,11.22±0.29,11.15±0.29,11.15±0.29,11.59±0.29,11.43±0.29,11.59±0.29,11.74±0.29,11.49±0.29,15.05±0.33,11.1±0.29,11.0±0.29,11.56±0.29,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,33.83±2.86,33.01±2.88,33.56±2.86,34.36±2.87,34.22±2.86,34.22±2.86,33.14±2.89,33.82±2.88,33.27±2.89,35.43±2.92,33.68±2.89,32.97±2.85,36.84±2.99,36.71±2.98,34.46±2.91,32.35±2.83


In [739]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]

In [740]:
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]

In [741]:
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))

/tmp/ipykernel_93986/1504875137.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [742]:
res_df.loc[8] = avg_res

In [743]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B,ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT,ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.43±2.79,29.78±2.78,30.67±2.84,30.52±2.83,30.15±2.82,29.81±2.82,30.79±2.83,30.55±2.82,30.61±2.82,30.17±2.81,30.56±2.82,32.39±2.87,28.98±2.71,28.94±2.74,30.12±2.81,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.75±3.07,26.77±3.15,27.78±3.19,29.8±3.26,28.28±3.21,28.28±3.21,23.74±3.03,24.24±3.05,24.75±3.07,24.75±3.07,24.75±3.07,25.76±3.12,23.23±3.01,22.73±2.99,25.25±3.1,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.09±1.86,26.37±1.89,25.64±1.87,23.99±1.83,22.16±1.78,22.71±1.79,26.37±1.89,26.74±1.9,26.56±1.89,26.19±1.88,27.11±1.9,22.53±1.79,25.82±1.87,26.74±1.9,25.82±1.87,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.23±2.11,23.66±2.01,29.69±2.16,29.46±2.16,29.24±2.15,29.69±2.16,23.88±2.02,25.0±2.05,23.88±2.02,24.33±2.03,24.78±2.04,27.46±2.11,25.22±2.05,26.34±2.08,24.11±2.02,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,19.41±1.7,8.69±1.21,14.23±1.5,12.75±1.44,13.31±1.46,13.12±1.45,9.43±1.26,10.54±1.32,9.61±1.27,11.09±1.35,10.54±1.32,26.25±1.89,13.68±1.48,12.38±1.42,14.05±1.5,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.11±0.11,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,1.34±0.61,0.0±0.0,0.0±0.0,0.1±0.07,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.19±0.29,11.54±0.29,11.17±0.29,11.22±0.29,11.15±0.29,11.15±0.29,11.59±0.29,11.43±0.29,11.59±0.29,11.74±0.29,11.49±0.29,15.05±0.33,11.1±0.29,11.0±0.29,11.56±0.29,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,33.83±2.86,33.01±2.88,33.56±2.86,34.36±2.87,34.22±2.86,34.22±2.86,33.14±2.89,33.82±2.88,33.27±2.89,35.43±2.92,33.68±2.89,32.97±2.85,36.84±2.99,36.71±2.98,34.46±2.91,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,19.79,18.30,19.57,19.77,19.52,19.43,18.11,18.43,18.31,18.86,18.50,22.29,18.97,18.63,19.26,27.76


In [744]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_93986/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [745]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B,ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT,ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44,30.75,29.43,29.78,30.67,30.52,30.15,29.81,30.79,30.55,30.61,30.17,30.56,32.39,28.98,28.94,30.12,33.46
1,gpqa_diamond,24.24,25.76,24.75,26.77,27.78,29.80,28.28,28.28,23.74,24.24,24.75,24.75,24.75,25.76,23.23,22.73,25.25,28.79
4,ifeval,10.17,12.20,19.41,8.69,14.23,12.75,13.31,13.12,9.43,10.54,9.61,11.09,10.54,26.25,13.68,12.38,14.05,50.28
5,math,0.12,0.32,0.11,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.34,0.00,0.00,0.10,5.53
6,mmlu_pro,11.79,11.05,11.19,11.54,11.17,11.22,11.15,11.15,11.59,11.43,11.59,11.74,11.49,15.05,11.10,11.00,11.56,16.17
7,musr,34.60,33.41,33.83,33.01,33.56,34.36,34.22,34.22,33.14,33.82,33.27,35.43,33.68,32.97,36.84,36.71,34.46,32.35
8,Avg. score (w. gpqa_diamond),18.73,18.92,19.79,18.30,19.57,19.77,19.52,19.43,18.11,18.43,18.31,18.86,18.50,22.29,18.97,18.63,19.26,27.76


## Extract OpenLM Leaderboard, Poster results

In [626]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin/template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin/template",
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-3s-Flan-Llama-3.2-1B",
    #"ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    "ARMT-16-64-DollyMix-5%-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [627]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [628]:
#overall_res

In [629]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()

In [630]:
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [631]:
res_df

,task,Llama-3.2-1B,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-16-64-3s-Flan-Llama-3.2-1B,ARMT-16-64-DollyMix-5%-8s-Llama-3.2-1B-IT,Llama-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.43±2.79,30.52±2.83,32.39±2.87,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.75±3.07,29.8±3.26,25.76±3.12,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.09±1.86,23.99±1.83,22.53±1.79,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.23±2.11,29.46±2.16,27.46±2.11,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,19.41±1.7,12.75±1.44,26.25±1.89,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.11±0.11,0.0±0.0,1.34±0.61,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.19±0.29,11.22±0.29,15.05±0.33,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,33.83±2.86,34.36±2.87,32.97±2.85,32.35±2.83


In [632]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]

In [633]:
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]

In [634]:
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))

/tmp/ipykernel_93986/1504875137.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [635]:
res_df.loc[8] = avg_res

In [636]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-16-64-3s-Flan-Llama-3.2-1B,ARMT-16-64-DollyMix-5%-8s-Llama-3.2-1B-IT,Llama-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.43±2.79,30.52±2.83,32.39±2.87,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.75±3.07,29.8±3.26,25.76±3.12,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.09±1.86,23.99±1.83,22.53±1.79,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.23±2.11,29.46±2.16,27.46±2.11,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,19.41±1.7,12.75±1.44,26.25±1.89,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.11±0.11,0.0±0.0,1.34±0.61,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.19±0.29,11.22±0.29,15.05±0.33,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,33.83±2.86,34.36±2.87,32.97±2.85,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,19.79,19.77,22.29,27.76


In [637]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_93986/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [643]:
res_df_sum["task"] = ["BBH", "GPQA (diamond)", "IFeval", "MATH", "MMLU Pro", "MUSR", "Avg. score"]

/tmp/ipykernel_93986/2260939022.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum["task"] = ["BBH", "GPQA (diamond)", "IFeval", "MATH", "MMLU Pro", "MUSR", "Avg. score"]


In [646]:
new_cols = {el: el for el in list(res_df_sum.columns)}
new_cols["task"] = "Task"

In [650]:
res_df_sum = res_df_sum.rename(new_cols, axis=1)

In [651]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
cmap = "BuGn"
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap).hide()

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

Task,Llama-3.2-1B,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-16-64-3s-Flan-Llama-3.2-1B,ARMT-16-64-DollyMix-5%-8s-Llama-3.2-1B-IT,Llama-3.2-1B-IT
BBH,31.44,30.75,29.43,30.52,32.39,33.46
GPQA (diamond),24.24,25.76,24.75,29.80,25.76,28.79
IFeval,10.17,12.20,19.41,12.75,26.25,50.28
MATH,0.12,0.32,0.11,0.00,1.34,5.53
MMLU Pro,11.79,11.05,11.19,11.22,15.05,16.17
MUSR,34.60,33.41,33.83,34.36,32.97,32.35
Avg. score,18.73,18.92,19.79,19.77,22.29,27.76


## Extract OpenLM Leaderboard, Loss masking ablation

In [780]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    "rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    "runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin/template",
    "data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    "runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin/template",
    
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    #"Flan-CT-Llama-3.2-1B",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    #"ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    #"ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Dolly-CompLoss-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B",
    
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [781]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [782]:
#overall_res

In [783]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [784]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_93986/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [785]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Dolly-CompLoss-2s-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.78±2.78,30.12±2.81,30.16±2.81,30.79±2.83,31.8±2.85,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,26.77±3.15,25.25±3.1,26.77±3.15,23.74±3.03,24.24±3.05,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,26.37±1.89,25.82±1.87,26.56±1.89,26.37±1.89,27.66±1.92,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,23.66±2.01,24.11±2.02,23.88±2.02,23.88±2.02,22.32±1.97,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,8.69±1.21,14.05±1.5,13.49±1.47,9.43±1.26,13.31±1.46,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.0±0.0,0.1±0.07,0.0±0.0,0.0±0.0,0.05±0.05,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.54±0.29,11.56±0.29,11.58±0.29,11.59±0.29,11.23±0.29,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,33.01±2.88,34.46±2.91,34.34±2.9,33.14±2.89,33.95±2.88,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,18.30,19.26,19.39,18.11,19.10,27.76


In [786]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_93986/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [787]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Dolly-CompLoss-2s-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44,30.75,29.78,30.12,30.16,30.79,31.80,33.46
1,gpqa_diamond,24.24,25.76,26.77,25.25,26.77,23.74,24.24,28.79
4,ifeval,10.17,12.20,8.69,14.05,13.49,9.43,13.31,50.28
5,math,0.12,0.32,0.00,0.10,0.00,0.00,0.05,5.53
6,mmlu_pro,11.79,11.05,11.54,11.56,11.58,11.59,11.23,16.17
7,musr,34.60,33.41,33.01,34.46,34.34,33.14,33.95,32.35
8,Avg. score (w. gpqa_diamond),18.73,18.92,18.30,19.26,19.39,18.11,19.10,27.76


## Extract OpenLM Leaderboard, Flan updates

In [788]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-98000/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64-inter/checkpoint-4000/pytorch_model.bin/template",
    "data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    #"Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    "Flan-CT-Llama-3.2-1B",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B",
    "ARMT-16-64-5s-Flan-CT-Llama-3.2-1B",
    "ARMT-16-64-8s-Flan-4k-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-CompLoss-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B",
    "ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B",
    
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [789]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [790]:
#overall_res

In [791]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [792]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_93986/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [793]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-Flan-4k-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,29.43±2.79,30.67±2.84,30.52±2.83,30.15±2.82,29.81±2.82,29.97±2.83,30.16±2.81,28.98±2.71,28.94±2.74,33.46±2.9
1,gpqa_diamond,24.24±3.05,24.75±3.07,27.78±3.19,29.8±3.26,28.28±3.21,28.28±3.21,28.79±3.23,26.77±3.15,23.23±3.01,22.73±2.99,28.79±3.23
2,gpqa_extended,24.54±1.84,25.09±1.86,25.64±1.87,23.99±1.83,22.16±1.78,22.71±1.79,23.08±1.8,26.56±1.89,25.82±1.87,26.74±1.9,25.09±1.86
3,gpqa_main,22.99±1.99,27.23±2.11,29.69±2.16,29.46±2.16,29.24±2.15,29.69±2.16,29.69±2.16,23.88±2.02,25.22±2.05,26.34±2.08,23.88±2.02
4,ifeval,10.17±1.3,19.41±1.7,14.23±1.5,12.75±1.44,13.31±1.46,13.12±1.45,13.49±1.47,13.49±1.47,13.68±1.48,12.38±1.42,50.28±2.15
5,math,0.12±0.12,0.11±0.11,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,5.53±1.56
6,mmlu_pro,11.79±0.29,11.19±0.29,11.17±0.29,11.22±0.29,11.15±0.29,11.15±0.29,11.29±0.29,11.58±0.29,11.1±0.29,11.0±0.29,16.17±0.34
7,musr,34.6±2.9,33.83±2.86,33.56±2.86,34.36±2.87,34.22±2.86,34.22±2.86,34.48±2.88,34.34±2.9,36.84±2.99,36.71±2.98,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,19.79,19.57,19.77,19.52,19.43,19.67,19.39,18.97,18.63,27.76


In [794]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_93986/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [795]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Flan-CT-Llama-3.2-1B,ARMT-16-64-3s-Flan-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B,ARMT-16-64-5s-Flan-CT-Llama-3.2-1B,ARMT-16-64-8s-Flan-4k-CT-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B,ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44,29.43,30.67,30.52,30.15,29.81,29.97,30.16,28.98,28.94,33.46
1,gpqa_diamond,24.24,24.75,27.78,29.80,28.28,28.28,28.79,26.77,23.23,22.73,28.79
4,ifeval,10.17,19.41,14.23,12.75,13.31,13.12,13.49,13.49,13.68,12.38,50.28
5,math,0.12,0.11,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5.53
6,mmlu_pro,11.79,11.19,11.17,11.22,11.15,11.15,11.29,11.58,11.10,11.00,16.17
7,musr,34.60,33.83,33.56,34.36,34.22,34.22,34.48,34.34,36.84,36.71,32.35
8,Avg. score (w. gpqa_diamond),18.73,19.79,19.57,19.77,19.52,19.43,19.67,19.39,18.97,18.63,27.76


## Extract OpenLM Leaderboard, Lora ablations

In [16]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    "rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    "runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    "runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    "runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin/template",

    "runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    "runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",

    "runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    
    "runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    "runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin/template",
    "runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-98000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64-inter/checkpoint-4000/pytorch_model.bin/template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000/pytorch_model.bin/template",
    "data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors/template",
    'runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin/template',
    
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    "Flan-CT-Llama-3.2-1B",

    "ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",

    "ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",

    "ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",

    "ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    ##"ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B",
    ##"ARMT-16-64-5s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-8s-Flan-4k-CT-Llama-3.2-1B",
    ##"ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-CompLoss-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B",
    "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",

    "ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"SmolTalk-CT-Llama-3.2-1B",
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [17]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [18]:
#overall_res

In [19]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [20]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_2170857/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [21]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B,ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.43±2.79,29.78±2.78,32.39±2.87,32.98±2.9,29.54±2.76,32.48±2.87,33.84±2.92,33.16±2.89,33.61±2.92,33.46±2.91,33.45±2.91,29.17±2.72,33.81±2.91,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.75±3.07,26.77±3.15,25.76±3.12,23.23±3.01,25.25±3.1,26.77±3.15,23.74±3.03,23.74±3.03,24.24±3.05,24.24±3.05,24.24±3.05,28.28±3.21,27.27±3.17,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.09±1.86,26.37±1.89,22.53±1.79,22.53±1.79,26.56±1.89,24.91±1.85,25.27±1.86,24.36±1.84,25.82±1.87,26.01±1.88,25.82±1.87,24.36±1.84,24.91±1.85,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.23±2.11,23.66±2.01,27.46±2.11,27.01±2.1,23.66±2.01,26.56±2.09,27.68±2.12,25.45±2.06,27.9±2.12,26.79±2.09,27.01±2.1,22.99±1.99,22.99±1.99,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,19.41±1.7,8.69±1.21,26.25±1.89,23.48±1.82,11.28±1.36,28.47±1.94,33.83±2.04,31.61±2.0,35.12±2.05,34.38±2.04,33.64±2.03,11.46±1.37,34.01±2.04,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.11±0.11,0.0±0.0,1.34±0.61,0.38±0.28,0.0±0.0,0.7±0.54,1.8±0.79,0.8±0.47,1.97±0.83,1.9±0.76,2.18±0.9,0.0±0.0,1.03±0.51,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.19±0.29,11.54±0.29,15.05±0.33,13.72±0.31,11.29±0.29,14.48±0.32,13.83±0.31,14.18±0.32,13.66±0.31,13.81±0.31,13.86±0.32,11.37±0.29,14.51±0.32,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,33.83±2.86,33.01±2.88,32.97±2.85,34.05±2.89,34.58±2.93,32.99±2.87,32.6±2.87,33.65±2.9,34.32±2.89,34.05±2.89,34.32±2.9,40.73±3.02,32.09±2.84,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,19.79,18.30,22.29,21.31,18.66,22.65,23.27,22.86,23.82,23.64,23.62,20.17,23.79,27.76


In [22]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_2170857/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [23]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B,ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44,30.75,29.43,29.78,32.39,32.98,29.54,32.48,33.84,33.16,33.61,33.46,33.45,29.17,33.81,33.46
1,gpqa_diamond,24.24,25.76,24.75,26.77,25.76,23.23,25.25,26.77,23.74,23.74,24.24,24.24,24.24,28.28,27.27,28.79
4,ifeval,10.17,12.20,19.41,8.69,26.25,23.48,11.28,28.47,33.83,31.61,35.12,34.38,33.64,11.46,34.01,50.28
5,math,0.12,0.32,0.11,0.00,1.34,0.38,0.00,0.70,1.80,0.80,1.97,1.90,2.18,0.00,1.03,5.53
6,mmlu_pro,11.79,11.05,11.19,11.54,15.05,13.72,11.29,14.48,13.83,14.18,13.66,13.81,13.86,11.37,14.51,16.17
7,musr,34.60,33.41,33.83,33.01,32.97,34.05,34.58,32.99,32.60,33.65,34.32,34.05,34.32,40.73,32.09,32.35
8,Avg. score (w. gpqa_diamond),18.73,18.92,19.79,18.30,22.29,21.31,18.66,22.65,23.27,22.86,23.82,23.64,23.62,20.17,23.79,27.76


## Extract OpenLM Leaderboard, IFT Comparison

In [926]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000/pytorch_model.bin/template",
    "/home/jovyan/gkuzmin/rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000/pytorch_model.bin/template",
    "/home/jovyan/gkuzmin/rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x8128_8128/checkpoint-1600/pytorch_model.bin/template",
    "/home/jovyan/gkuzmin/rmt_ift_comparison/ift_from_base_hf_trainer_big_lr/dolly/meta-llama/rmt_ift_comparison/ift_from_base_hf_trainer_big_lr/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-100/pytorch_model.bin/template",
    #"../rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_8x1024_8128/checkpoint-1600/template",
    
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    "Flan-CT-Llama-3.2-1B",


    "SmolTalk-CT-Llama-3.2-1B",
    "SmolTalk-8k-CT-Llama-3.2-1B",
    "Dolly-BigLR-CT-Llama-3.2-1B",

    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [927]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    if model_path.startswith("/"):
        buf_path = model_path
    else:
        buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [928]:
#overall_res

In [929]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [930]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_93986/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [931]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,SmolTalk-CT-Llama-3.2-1B,SmolTalk-8k-CT-Llama-3.2-1B,Dolly-BigLR-CT-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,29.43±2.79,29.72±2.8,30.41±2.82,31.34±2.87,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.75±3.07,25.25±3.1,32.32±3.33,23.74±3.03,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.09±1.86,24.54±1.84,27.66±1.92,26.19±1.88,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.23±2.11,25.22±2.05,26.34±2.08,22.54±1.98,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,19.41±1.7,26.43±1.9,25.88±1.88,11.09±1.35,50.28±2.15
5,math,0.12±0.12,0.32±0.29,0.11±0.11,0.25±0.21,0.0±0.0,0.35±0.29,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,11.19±0.29,11.69±0.29,11.87±0.29,11.21±0.29,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,33.83±2.86,34.08±2.9,31.83±2.84,35.41±2.94,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,19.79,21.24,22.05,18.86,27.76


In [932]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_93986/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [933]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,Flan-CT-Llama-3.2-1B,SmolTalk-CT-Llama-3.2-1B,SmolTalk-8k-CT-Llama-3.2-1B,Dolly-BigLR-CT-Llama-3.2-1B,Llama-CT-3.2-1B-IT
0,bbh,31.44,30.75,29.43,29.72,30.41,31.34,33.46
1,gpqa_diamond,24.24,25.76,24.75,25.25,32.32,23.74,28.79
4,ifeval,10.17,12.20,19.41,26.43,25.88,11.09,50.28
5,math,0.12,0.32,0.11,0.25,0.00,0.35,5.53
6,mmlu_pro,11.79,11.05,11.19,11.69,11.87,11.21,16.17
7,musr,34.60,33.41,33.83,34.08,31.83,35.41,32.35
8,Avg. score (w. gpqa_diamond),18.73,18.92,19.79,21.24,22.05,18.86,27.76


## Extract OpenLM Leaderboard, Babilong models

In [3]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    # "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",

    #"runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    
    #"runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin/template",
    "runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-98000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64-inter/checkpoint-4000/pytorch_model.bin/template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000/pytorch_model.bin/template",
    # "data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors/template",
    # "data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors/template",
    
    "runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-500/pytorch_model.bin/template",
    "runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin/template",
    "runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    # "runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",

    "runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin/template",

    "runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",

    
    # "runs/armt_ift_from_babilong/smol_lora_comp_bl_4s_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-10000/pytorch_model.bin/template",
    #'runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin/template',


    #"runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-7500/pytorch_model.bin/template",
    #"runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-7500/pytorch_model.bin/template",
    "it_on_base/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5_ct-v3.1/model.safetensors/template",


    "data/pretrained_models/RMT-Llama-3.2-1B-Instruct-2x1024-mem16-lora-smol_qa1-5-1_9-qa1-5-distill_1.0-1000_steps/model.safetensors/template",
    "data/pretrained_models/RMT-Llama-3.2-1B-Instruct-2x1024-mem16-lora-dolly_qa1-5-1_9-qa1-5-distill_1.0-1000_steps/model.safetensors/template",

    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    # "Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    #"Flan-CT-Llama-3.2-1B",

    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",

    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    ##"ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B",
    ##"ARMT-16-64-5s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-8s-Flan-4k-CT-Llama-3.2-1B",
    ##"ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-CompLoss-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B",
    # "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    # "ARMT-16-64-4s-CL-BabiLong-QA1-5_v3-Lora-CompLoss-Llama-3.2-1B-IT",
    
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-1000-CompLoss-Llama-3.2-1B-IT",
    # "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora1e4-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT",

    "ARMT-16-64-8s-2k-synth_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT",
    # "ARMT-32-64-4s-Smol_from_BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT",


    #"ARMT-16-64-4s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"SmolTalk-CT-Llama-3.2-1B",
    "ARMT-16-64-8s-CL-BabiLong-QA1-5_v3.1-Lora-CompLoss-Llama-3.2-1B-IT",

    "ARMT-16-64-2s-CL-DollyDistil-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmolDistil-Llama-3.2-1B-IT",

    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [4]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [5]:
#overall_res

In [6]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [7]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_2656816/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [8]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-1000-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-2k-synth_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-BabiLong-QA1-5_v3.1-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-DollyDistil-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmolDistil-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,33.45±2.91,32.72±2.9,32.29±2.88,31.09±2.86,31.86±2.89,30.5±2.85,29.35±2.72,33.15±2.89,33.45±2.9,33.46±2.9
1,gpqa_diamond,24.24±3.05,24.24±3.05,25.25±3.1,23.23±3.01,24.24±3.05,27.78±3.19,21.72±2.94,26.77±3.15,29.8±3.26,28.79±3.23,28.79±3.23
2,gpqa_extended,24.54±1.84,25.82±1.87,23.44±1.81,23.08±1.8,20.33±1.72,23.26±1.81,23.81±1.82,26.92±1.9,25.46±1.87,26.01±1.88,25.09±1.86
3,gpqa_main,22.99±1.99,27.01±2.1,24.33±2.03,24.33±2.03,25.22±2.05,26.56±2.09,29.46±2.16,24.55±2.04,23.66±2.01,23.88±2.02,23.88±2.02
4,ifeval,10.17±1.3,33.64±2.03,22.55±1.8,23.11±1.81,10.54±1.32,19.04±1.69,1.85±0.58,10.17±1.3,26.8±1.91,29.21±1.96,50.28±2.15
5,math,0.12±0.12,2.18±0.9,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,5.53±1.56
6,mmlu_pro,11.79±0.29,13.86±0.32,11.89±0.3,12.55±0.3,10.8±0.28,10.75±0.28,10.9±0.28,11.28±0.29,14.3±0.32,14.23±0.32,16.17±0.34
7,musr,34.6±2.9,34.32±2.9,32.61±2.85,33.01±2.87,33.83±2.88,32.07±2.83,35.81±2.93,40.48±2.99,31.02±2.8,31.29±2.81,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,23.62,20.84,20.70,18.42,20.25,16.80,19.68,22.51,22.83,27.76


In [9]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_2656816/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [13]:

cols = res_df_sum.columns[[0,-3,-2,-1]]
res_df_sum = res_df_sum[cols]

In [14]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,ARMT-16-64-2s-CL-DollyDistil-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmolDistil-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,33.15,33.45,33.46
1,gpqa_diamond,29.80,28.79,28.79
4,ifeval,26.80,29.21,50.28
5,math,0.00,0.00,5.53
6,mmlu_pro,14.30,14.23,16.17
7,musr,31.02,31.29,32.35
8,Avg. score (w. gpqa_diamond),22.51,22.83,27.76


## Extract OpenLM Leaderboard, new Smoltalk models

In [107]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",

    #"runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    
    #"runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin/template",
    "runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    
    
    #"data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors/template",
    #"data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors/template",
    #"it_on_base/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5_ct-v3.1/model.safetensors/template",
    #"runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin/template",
    #"runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",

    
    #"runs/armt_ift_from_babilong/smol_lora_comp_bl_4s_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-10000/pytorch_model.bin/template",
    #'runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin/template',


    "runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-7500/pytorch_model.bin/template",
    "runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-7500/pytorch_model.bin/template",

    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    #"Flan-CT-Llama-3.2-1B",

    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",

    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    
    
    #"ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-BabiLong-QA1-5_v3-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-BabiLong-QA1-5_v3.1-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora1e4-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-4s-Smol_from_BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT",


    "ARMT-16-64-4s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"SmolTalk-CT-Llama-3.2-1B",
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [108]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [109]:
#overall_res

In [110]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [111]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_5170/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [112]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,33.45±2.91,33.13±2.88,32.92±2.86,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.24±3.05,25.76±3.12,24.75±3.07,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.82±1.87,26.37±1.89,26.92±1.9,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.01±2.1,24.78±2.04,24.55±2.04,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,33.64±2.03,28.1±1.93,26.62±1.9,50.28±2.15
5,math,0.12±0.12,0.32±0.29,2.18±0.9,0.0±0.0,0.0±0.0,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,13.86±0.32,14.12±0.32,15.3±0.33,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,34.32±2.9,34.34±2.89,33.94±2.91,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,23.62,22.58,22.26,27.76


In [113]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_5170/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [114]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-SmolMix05-noLora-CompLoss-Llama-3.2-1B-IT,Llama-CT-3.2-1B-IT
0,bbh,31.44,30.75,33.45,33.13,32.92,33.46
1,gpqa_diamond,24.24,25.76,24.24,25.76,24.75,28.79
4,ifeval,10.17,12.20,33.64,28.10,26.62,50.28
5,math,0.12,0.32,2.18,0.00,0.00,5.53
6,mmlu_pro,11.79,11.05,13.86,14.12,15.30,16.17
7,musr,34.60,33.41,34.32,34.34,33.94,32.35
8,Avg. score (w. gpqa_diamond),18.73,18.92,23.62,22.58,22.26,27.76


## Extract OpenLM Leaderboard, Loss ablation models, LoRA vs no-LoRA

In [52]:
base_path = "../../harness_results"
model_pathes = [
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-no-template/meta-llama__Llama-3.2-1B",
    "hf_leaderboard/meta-llama/Llama-3.2-1B/dolly_1k/checkpoint-700-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__ift_from_base_hf_trainer__dolly__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-700",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/sgpt_1k/checkpoint-28500-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__sharegpt__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-28500",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B/flan_1k/checkpoint-91000-template/__home__jovyan__gkuzmin__rmt_it__rmt_ift_comparison__full_ift_from_base_hf_trainer__flan__meta-llama__Llama-3.2-1B__SEGM_1x1024_1024__checkpoint-91000",

    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",

    #"runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    
    #"runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin/template",
    "runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin/template",
    #"rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-98000/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64-inter/checkpoint-4000/pytorch_model.bin/template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin/no-template",
    #"data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin/no-template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin/template",
    #"runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin/template",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000/pytorch_model.bin/template",
    #"data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors/template",
    #"data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors/template",
    #"runs/armt_ift_from_babilong/smol_lora_comp_bl_4s_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-10000/pytorch_model.bin/template",
    #'runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin/template',
    "runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-10000/pytorch_model.bin/template",
    "runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9700/pytorch_model.bin/template",
    #"runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9500/pytorch_model.bin/template",
    #"runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin/template",

    "it_on_base/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9500/pytorch_model.bin/template",
    "it_on_base/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin/template",
    "runs/loss_abaltion_no_lora_ext/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64_1e-05_100000/checkpoint-88000/pytorch_model.bin/template",
    
    "hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "Llama-3.2-1B",
    "Dolly-CT-Llama-3.2-1B",
    #"SGPT-CT-Llama-3.2-1B",
    #"Flan-CT-Llama-3.2-1B",

    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",

    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-8s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CompLoss-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-SGPT-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-ShortSynth-CT-Llama-3.2-1B",
    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B-IT",
    ##"ARMT-16-64-2s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-3s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-5s-Flan-48k-CT-Llama-3.2-1B",
    ##"ARMT-16-64-5s-Flan-CT-Llama-3.2-1B",
    ##"ARMT-16-64-8s-Flan-4k-CT-Llama-3.2-1B",
    ##"ARMT-16-64-2s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Dolly-CompLoss-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-31500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-49500-Dolly-2s-Llama-3.2-1B",
    #"ARMT-16-64-4s-LM-19500-Dolly-4s-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-16-64-2s-DollyMix05-8s-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-22k-Llama-3.2-1B",
    #"ARMT-16-64-2s-LM-19500-Flan-2s-Llama-3.2-1B",
    #"ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-BabiLong-QA1-5_v3-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-4s-Smol_from_BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"SmolTalk-CT-Llama-3.2-1B",
    "ARMT-16-64-2s-Synth2k-1m-Lora-CompLoss-Llama-3.2-1B",
    "ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-16-64-2s-Synth2k-1m-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B-IT",

    "ARMT-16-64-2s-Synth2k-1m-Lora-CompLoss-Llama-3.2-1B-IT-from-IT",
    "ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B-IT-from-IT",
    "ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B-100k",

    
    
    "Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [53]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [54]:
#overall_res

In [55]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [56]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_5170/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [57]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-Synth2k-1m-Lora-CompLoss-Llama-3.2-1B,ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B,ARMT-16-64-2s-Synth2k-1m-Lora-CompLoss-Llama-3.2-1B-IT-from-IT,ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B-IT-from-IT,ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B-100k,Llama-CT-3.2-1B-IT
0,bbh,31.44±2.74,30.75±2.83,33.45±2.91,29.72±2.81,30.05±2.81,29.95±2.82,31.97±2.87,30.03±2.78,33.46±2.9
1,gpqa_diamond,24.24±3.05,25.76±3.12,24.24±3.05,24.75±3.07,26.26±3.14,24.75±3.07,28.28±3.21,26.77±3.15,28.79±3.23
2,gpqa_extended,24.54±1.84,24.73±1.85,25.82±1.87,27.11±1.9,26.92±1.9,26.74±1.9,24.91±1.85,24.18±1.83,25.09±1.86
3,gpqa_main,22.99±1.99,25.45±2.06,27.01±2.1,24.11±2.02,24.33±2.03,25.89±2.07,21.43±1.94,22.1±1.96,23.88±2.02
4,ifeval,10.17±1.3,12.2±1.41,33.64±2.03,16.82±1.61,19.22±1.7,29.94±1.97,31.61±2.0,15.9±1.57,50.28±2.15
5,math,0.12±0.12,0.32±0.29,2.18±0.9,0.0±0.0,0.0±0.0,0.0±0.0,0.0±0.0,0.14±0.14,5.53±1.56
6,mmlu_pro,11.79±0.29,11.05±0.29,13.86±0.32,10.91±0.28,10.85±0.28,11.53±0.29,14.84±0.32,11.49±0.29,16.17±0.34
7,musr,34.6±2.9,33.41±2.89,34.32±2.9,37.81±2.95,36.75±2.93,34.88±2.89,32.61±2.87,35.16±2.92,32.35±2.83
8,Avg. score (w. gpqa_diamond),18.73,18.92,23.62,20.00,20.52,21.84,23.22,19.92,27.76


In [58]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_5170/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [59]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,Llama-3.2-1B,Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-Synth2k-1m-Lora-CompLoss-Llama-3.2-1B,ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B,ARMT-16-64-2s-Synth2k-1m-Lora-CompLoss-Llama-3.2-1B-IT-from-IT,ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B-IT-from-IT,ARMT-16-64-2s-Synth2k-1m-no-Lora-CompLoss-Llama-3.2-1B-100k,Llama-CT-3.2-1B-IT
0,bbh,31.44,30.75,33.45,29.72,30.05,29.95,31.97,30.03,33.46
1,gpqa_diamond,24.24,25.76,24.24,24.75,26.26,24.75,28.28,26.77,28.79
4,ifeval,10.17,12.20,33.64,16.82,19.22,29.94,31.61,15.90,50.28
5,math,0.12,0.32,2.18,0.00,0.00,0.00,0.00,0.14,5.53
6,mmlu_pro,11.79,11.05,13.86,10.91,10.85,11.53,14.84,11.49,16.17
7,musr,34.60,33.41,34.32,37.81,36.75,34.88,32.61,35.16,32.35
8,Avg. score (w. gpqa_diamond),18.73,18.92,23.62,20.00,20.52,21.84,23.22,19.92,27.76


## Extract OpenLM Leaderboard, SFT for ARMT-SmolLM-135M

In [3]:
base_path = "../../harness_results"
model_pathes = [
    "armt_hf_leaderboard/HuggingFaceTB/SmolLM2-135M-Instruct/template",
    "runs/inst_pretrain_armt_sw/armt_1k_8s_smollm/smol_smoltalk/HuggingFaceTB/SmolLM2-135M/mask_fix_assist_mask_no_sw_no_lora_SEGM_8x1024_8064_64_3e-04_1000/checkpoint-1000/pytorch_model.bin/template",
    "armt_hf_leaderboard/HuggingFaceTB/SmolLM2-135M/no-template",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "SmolLM2-135M-Instruct-CT",
    "ARMT-SmolLM2-135M-Instruct-8*1024-CT",
    "SmolLM2-135M",
    
    #"Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [4]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [5]:
#overall_res

In [6]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [7]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_39495/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [8]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,SmolLM2-135M-Instruct-CT,ARMT-SmolLM2-135M-Instruct-8*1024-CT,SmolLM2-135M
0,bbh,30.39±2.8,30.9±2.81,30.22±2.71
1,gpqa_diamond,26.77±3.15,27.78±3.19,26.26±3.14
2,gpqa_extended,25.27±1.86,28.02±1.92,27.29±1.91
3,gpqa_main,23.44±2.0,23.88±2.02,21.43±1.94
4,ifeval,21.63±1.77,12.2±1.41,11.65±1.38
5,math,0.23±0.23,0.0±0.0,0.05±0.05
6,mmlu_pro,10.96±0.28,10.48±0.28,10.9±0.28
7,musr,37.26±2.97,37.66±3.0,41.38±3.04
8,Avg. score (w. gpqa_diamond),21.21,19.84,20.08


In [9]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_39495/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [10]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,SmolLM2-135M-Instruct-CT,ARMT-SmolLM2-135M-Instruct-8*1024-CT,SmolLM2-135M
0,bbh,30.39,30.90,30.22
1,gpqa_diamond,26.77,27.78,26.26
4,ifeval,21.63,12.20,11.65
5,math,0.23,0.00,0.05
6,mmlu_pro,10.96,10.48,10.90
7,musr,37.26,37.66,41.38
8,Avg. score (w. gpqa_diamond),21.21,19.84,20.08


## Extract OpenLM Leaderboard, SFT for ARMT-SmolLM-360M + PT

In [3]:
base_path = "../../harness_results"
model_pathes = [
    "armt_hf_leaderboard/HuggingFaceTB/SmolLM2-360M-Instruct/template",
    "armt_hf_leaderboard/HuggingFaceTB/SmolLM2-360M/no-template",
    "armt_hf_leaderboard/HuggingFaceTB/SmolLM2-360M-Instruct/no-template",
    #"runs/inst_pretrain_armt_sw/armt_1k_8s_smollm/smol_smoltalk/HuggingFaceTB/SmolLM2-135M/mask_fix_assist_mask_no_sw_no_lora_SEGM_8x1024_8064_64_3e-04_1000/checkpoint-1000/pytorch_model.bin/template",
    #"armt_hf_leaderboard/HuggingFaceTB/SmolLM2-135M/no-template",
    #"hf_leaderboard/meta-llama/Llama-3.2-1B-Instruct/base-template/unsloth__Llama-3.2-1B-Instruct",
]
model_names = [
    "SmolLM2-360M-Instruct-CT",
    "SmolLM2-360M",
    "SmolLM2-360M-Instruct",
    #"ARMT-SmolLM2-135M-Instruct-8*1024-CT",
    #"SmolLM2-135M",
    
    #"Llama-CT-3.2-1B-IT",
]
task_keys = ["mmlu", "arc_challenge", "arc_easy", "hellaswag", "mathqa"]
task_metrics = [["acc,none", "acc_stderr,none"]]*5#, "acc,none", "acc,none", "acc,none", "acc,none"]

task_keys = ['leaderboard_bbh_boolean_expressions', 'leaderboard_bbh_causal_judgement', 'leaderboard_bbh_date_understanding', 
             'leaderboard_bbh_disambiguation_qa', 'leaderboard_bbh_formal_fallacies', 'leaderboard_bbh_geometric_shapes', 'leaderboard_bbh_hyperbaton',
             'leaderboard_bbh_logical_deduction_five_objects', 'leaderboard_bbh_logical_deduction_seven_objects',
             'leaderboard_bbh_logical_deduction_three_objects', 'leaderboard_bbh_movie_recommendation', 'leaderboard_bbh_navigate',
             'leaderboard_bbh_object_counting', 'leaderboard_bbh_penguins_in_a_table', 'leaderboard_bbh_reasoning_about_colored_objects',
             'leaderboard_bbh_ruin_names', 'leaderboard_bbh_salient_translation_error_detection', 'leaderboard_bbh_snarks',
             'leaderboard_bbh_sports_understanding', 'leaderboard_bbh_temporal_sequences', 'leaderboard_bbh_tracking_shuffled_objects_five_objects',
             'leaderboard_bbh_tracking_shuffled_objects_seven_objects', 'leaderboard_bbh_tracking_shuffled_objects_three_objects', 'leaderboard_bbh_web_of_lies',
             'leaderboard_gpqa_diamond', 'leaderboard_gpqa_extended', 'leaderboard_gpqa_main', 'leaderboard_ifeval',
             'leaderboard_math_algebra_hard', 'leaderboard_math_counting_and_prob_hard', 'leaderboard_math_geometry_hard',
             'leaderboard_math_intermediate_algebra_hard', 'leaderboard_math_num_theory_hard', 'leaderboard_math_prealgebra_hard',
             'leaderboard_math_precalculus_hard', 'leaderboard_mmlu_pro', 'leaderboard_musr_murder_mysteries',
             'leaderboard_musr_object_placements', 'leaderboard_musr_team_allocation']
task_metrics = [["acc_norm,none", "acc_norm_stderr,none"]]*24 + [["acc_norm,none", "acc_norm_stderr,none"]]*3 +\
                [["prompt_level_strict_acc,none", "prompt_level_strict_acc_stderr,none"]] +\
                [["exact_match,none", "exact_match_stderr,none"]]*7 + [["acc,none", "acc_stderr,none"]]*1 +\
                [["acc_norm,none", "acc_norm_stderr,none"]]*3

groups = [{"bbh":24}, {"gpqa_diamond":1}, {"gpqa_extended":1}, {"gpqa_main":1}, {"ifeval":1}, {"math":7}, {"mmlu_pro":1}, {"musr":3}]


In [4]:
overall_res = {}
for model_name, model_path in zip(model_names, model_pathes):
    buf_path = os.path.join(base_path, model_path)
    buf_res = load_results(buf_path, task_keys, task_metrics, groups)
    overall_res[model_name] = buf_res

In [5]:
#overall_res

In [6]:
res_df = pd.DataFrame.from_dict(overall_res).sort_index()
res_df.reset_index(inplace=True)
res_df = res_df.rename({"index": "task"}, axis=1)

In [7]:
rows_to_sum = ["bbh", "gpqa_diamond", "ifeval", "math", "mmlu_pro", "musr"]
res_df_sum = res_df[res_df["task"].isin(rows_to_sum)]
model_cols = list(res_df_sum.columns)[1:]
#pd.concat([res_df_sum, ["average"] + [0]*len(model_cols)])
for col in model_cols:
    res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))
avg_res = ["Avg. score (w. gpqa_diamond)"] + list(np.round(res_df_sum[model_cols].mean().values, 2))
res_df.loc[8] = avg_res

/tmp/ipykernel_9130/2741836849.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum[col] = res_df_sum[col].apply(lambda x: float(x.split("±")[0]))


In [8]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,SmolLM2-360M-Instruct-CT,SmolLM2-360M,SmolLM2-360M-Instruct
0,bbh,31.63±2.8,31.2±2.8,30.23±2.79
1,gpqa_diamond,27.27±3.17,26.26±3.14,24.24±3.05
2,gpqa_extended,24.54±1.84,26.37±1.89,24.91±1.85
3,gpqa_main,23.66±2.01,22.32±1.97,22.1±1.96
4,ifeval,31.24±1.99,12.75±1.44,6.84±1.09
5,math,1.73±0.86,1.27±0.75,1.26±0.73
6,mmlu_pro,11.04±0.29,11.69±0.29,11.05±0.29
7,musr,34.61±2.9,39.94±3.0,34.23±2.87
8,Avg. score (w. gpqa_diamond),22.92,20.52,17.97


In [9]:
res_df_sum.loc[8] = avg_res

/tmp/ipykernel_9130/2501945193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_df_sum.loc[8] = avg_res


In [10]:
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = res_df_sum.style
for l0 in [7]:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
#fres = s.format(precision=2)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)

#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,task,SmolLM2-360M-Instruct-CT,SmolLM2-360M,SmolLM2-360M-Instruct
0,bbh,31.63,31.20,30.23
1,gpqa_diamond,27.27,26.26,24.24
4,ifeval,31.24,12.75,6.84
5,math,1.73,1.27,1.26
6,mmlu_pro,11.04,11.69,11.05
7,musr,34.61,39.94,34.23
8,Avg. score (w. gpqa_diamond),22.92,20.52,17.97
